In [2]:
using SpeedyWeather, CairoMakie, GLMakie

# Define grid and model type
spectral_grid = SpectralGrid()
model = PrimitiveWetModel(spectral_grid)

# Initialise the model and check its ouput
simulation = initialize!(model)
model.output

NetCDFOutput{Field{Float32, 1, Vector{Float32}, FullGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}}}
├ status: inactive/uninitialized
├ write restart file: true (if active)
├ interpolator: AnvilInterpolator{Float32, GridGeometry{OctahedralGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}, Vector{Float32}, Vector{Int64}}, AnvilLocator{Float32, Vector{Float32}, Vector{Int64}}}
├ path: output.nc (overwrite=false)
├ frequency: 21600 seconds
└┐ variables:
 ├ v: meridional wind [m/s]
 ├ humid: specific humidity [kg/kg]
 ├ temp: temperature [degC]
 ├ u: zonal wind [m/s]
 ├ mslp: mean sea-level pressure [hPa]
 └ vor: relative vorticity [s^-1]

In [3]:
# Add radiation and surface flux to the model
add!(model, SpeedyWeather.RadiationOutput()...)
add!(model, SpeedyWeather.SurfaceFluxesOutput()...)

NetCDFOutput{Field{Float32, 1, Vector{Float32}, FullGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}}}
├ status: inactive/uninitialized
├ write restart file: true (if active)
├ interpolator: AnvilInterpolator{Float32, GridGeometry{OctahedralGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}, Vector{Float32}, Vector{Int64}}, AnvilLocator{Float32, Vector{Float32}, Vector{Int64}}}
├ path: output.nc (overwrite=false)
├ frequency: 21600 seconds
└┐ variables:
 ├ slf: Surface latent heat flux (positive up) [W/m^2]
 ├ sru: Surface shortwave radiation up [W/m^2]
 ├ temp: temperature [degC]
 ├ srd: Surface shortwave radiation down [W/m^2]
 ├ mslp: mean sea-level pressure [hPa]
 ├ vor: relative vorticity [s^-1]
 ├ osr: Outgoing shortwave radiation [W/m^2]
 ├ v: meridional wind [m/s]
 ├ u: zonal wind [m/s]
 ├ albedo: albedo [1]
 ├ lrd: Surface longwave radiation down [W/m^2]
 ├ shf: Surface sensible heat flux (positive up) [W/m^2]
 

In [4]:
# Check radiation and surface flux types
simulation.diagnostic_variables.physics.sensible_heat_flux
simulation.diagnostic_variables.physics.surface_latent_heat_flux

3168-element, 48-ring OctahedralGaussianField{Float32, 1} as Array on CPU
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 ⋮
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0
 0.0f0

In [5]:
# Run simulation
run!(simulation, period=Day(365))

In [ ]:
field = simulation.diagnostic_variables.physics.sensible_heat_flux
g  = field.grid
vals = Array(field)   # the 1D field values (length 3168)
heatmap(field)

In [121]:
field

3168-element, 48-ring OctahedralGaussianField{Float32, 1} as Array on CPU
  2.022669f0
  0.15676351f0
  0.0f0
  0.0f0
  0.0f0
  0.0f0
  0.0f0
  7.2197495f0
 16.89863f0
 21.239723f0
  ⋮
  0.0f0
  0.0f0
  0.0f0
  0.0f0
  0.0f0
  0.0f0
  0.0f0
  0.0f0
  0.0f0

In [167]:
full_field = RingGrids.interpolate(FullGaussianGrid, field)
full_field_matrix = Matrix(full_field)

96×48 Matrix{Float32}:
  2.0227       93.2981   31.1369    …   0.0      0.0  0.0  0.0  0.0
  1.63397      92.3128   23.9475        1.37441  0.0  0.0  0.0  0.0
  1.24524      91.3275   16.7581        2.74882  0.0  0.0  0.0  0.0
  0.856507     90.3422    9.56863       6.42049  0.0  0.0  0.0  0.0
  0.467777     89.3569    8.28095      14.6867   0.0  0.0  0.0  0.0
  0.15026      85.1629   11.4196    …  22.9529   0.0  0.0  0.0  0.0
  0.117599     80.9689   14.5582       19.2812   0.0  0.0  0.0  0.0
  0.0849387    76.7749   17.5282        9.6406   0.0  0.0  0.0  0.0
  0.0522783    72.5808   19.4863        0.0      0.0  0.0  0.0  0.0
  0.0196172    66.2301   21.4443        0.0      0.0  0.0  0.0  0.0
  ⋮                                 ⋱                 ⋮         
 -2.58631f-6   -7.87722   0.0          23.9155   0.0  0.0  0.0  0.0
 -3.44841f-6  -10.503     0.0           0.0      0.0  0.0  0.0  0.0
  3.5441f-6    10.7944    0.0           0.0      0.0  0.0  0.0  0.0
  1.05366f-5   32.0918    0.

In [ ]:
using CairoMakie, GeoMakie

fig = Figure(size = (1200, 600))

# GeoAxis: sets up a geographic axis so coastlines and lon/lat align.
ga = GeoAxis(fig[1, 1];
             xlabel = "Longitude", ylabel = "Latitude")

# scatter!(ga, lon, lat;
#         color = vals,            
#         colormap = :thermal,
#         markersize = 15,
#         marker = :rect,
#         strokewidth = 0)

surface!(ga, full_field_matrix)

# Draw coastlines on top 
lines!(ga, GeoMakie.coastlines(); linewidth = 1, color="black")

# Add colourbar
Colorbar(fig[1, 2], label = "sensible heat flux")
fig


In [29]:
# Add a heatmap to see global patterns
heatmap(simulation.diagnostic_variables.physics.sensible_heat_flux)
#simulation.prognostic_variables.clock